In [1]:
import pandas as pd

In [2]:
dfp= pd.read_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/raw/HungRumEslo.csv')

In [3]:
dfp.head(2)

,name,essround,edition,proddate,idno,cntry,dweight,pspwght,pweight,anweight,...,trstlgl,trstplc,trstplt,trstprl,trstprt,trstsci,imbgeco,imwbcnt,happy,rlgdgr
0,ESS1e06_7,1,6.7,23.11.2023,2,HU,1.0,1.156497,0.479563,0.554613,...,5,5,6,7,NaN,NaN,5,5,8,5
1,ESS1e06_7,1,6.7,23.11.2023,3,HU,1.0,1.011243,0.479563,0.484955,...,4,3,4,8,NaN,NaN,3,7,7,5


In [4]:
#Eliminamos columnas del df que no interesen
columnas_a_eliminar = ['name', 'essround', 'edition', 'proddate', 'idno', 'dweight', 'pspwght', 'pweight', 'anweight', 'prob', 'stratum', 'psu']
dfp.drop(columnas_a_eliminar, axis=1, inplace=True)

In [5]:
#Obtenemos una lista con los nombres de las columnas restantes.
nombres_columnas = dfp.columns.tolist()
print(nombres_columnas)

['cntry', 'pplfair', 'pplhlp', 'ppltrst', 'lrscale', 'polintr', 'stfdem', 'stfeco', 'stfgov', 'trstep', 'trstlgl', 'trstplc', 'trstplt', 'trstprl', 'trstprt', 'trstsci', 'imbgeco', 'imwbcnt', 'happy', 'rlgdgr']


In [6]:
#Obtenemos una lista con los paises de las muestras de las filas.
valores_unicos_cntry = dfp['cntry'].unique()
print(valores_unicos_cntry)

['HU' 'SK']


In [7]:
#Guardamos en un dataframe nuestra tabla para asignar grupo y valores clave de la columna cntry.
import sqlite3
import pandas as pd


db_path = 'C:/Users/Josue/4GA.Datascience/4GA.DataScience/src/EcoUE.db'
try:
    conn = sqlite3.connect(db_path)
    query = "SELECT * FROM ecoeu"
    df_ecoeu = pd.read_sql_query(query, conn)
    print(df_ecoeu)
except sqlite3.Error as e:
    print(f"Error al conectar o consultar la base de datos: {e}")

finally:
    
    if conn:
        conn.close()

           cntry       PIB  Inflation    sma       cntrycat  color  \
0       Alemania   54343.2        5.9  60867           Rico  green   
1        Austria   56033.6        7.8  57082           Rico  green   
2        Bélgica   54700.9        4.0  59285           Rico  green   
3         Chipre   36551.4        3.5  26689  Media Europea   blue   
4        Croacia   21865.5        7.9  17714  Media Europea   blue   
5      Eslovenia   32610.1        7.4  26667  Media Europea   blue   
6         España   33509.0        3.5  30237  Media Europea   blue   
7        Estonia   30133.3        9.2  21595  Media Europea   blue   
8      Finlandia   52925.7        6.3  53310           Rico  green   
9        Francia   44690.9        4.9  43438  Media Europea   blue   
10        Grecia   23400.7        3.5  23536  Media Europea   blue   
11       Irlanda  103887.8        6.3  59899           Rico  green   
12        Italia   39003.3        5.6  33492  Media Europea   blue   
13       Letonia   2

In [8]:
cntrymap= {
    'HU':'Hungría',
    'SK':'Eslovaquia', 
}
dfp['cntry'] = dfp['cntry'].map(cntrymap)
dfp = pd.merge(dfp, df_ecoeu[['cntry', 'cntrycat_factorizado']], on='cntry', how='left')

In [9]:
print(dfp[['cntry', 'cntrycat_factorizado']])

            cntry  cntrycat_factorizado
0         Hungría                     2
1         Hungría                     2
2         Hungría                     2
3         Hungría                     2
4         Hungría                     2
...           ...                   ...
31489  Eslovaquia                     2
31490  Eslovaquia                     2
31491  Eslovaquia                     2
31492  Eslovaquia                     2
31493  Eslovaquia                     2

[31494 rows x 2 columns]


In [10]:
dfp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31494 entries, 0 to 31493
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   cntry                 31494 non-null  object 
 1   pplfair               31494 non-null  int64  
 2   pplhlp                31494 non-null  int64  
 3   ppltrst               31494 non-null  int64  
 4   lrscale               31494 non-null  int64  
 5   polintr               31494 non-null  int64  
 6   stfdem                31494 non-null  int64  
 7   stfeco                31494 non-null  int64  
 8   stfgov                31494 non-null  int64  
 9   trstep                31494 non-null  int64  
 10  trstlgl               31494 non-null  int64  
 11  trstplc               31494 non-null  int64  
 12  trstplt               31494 non-null  int64  
 13  trstprl               31494 non-null  int64  
 14  trstprt               29809 non-null  float64
 15  trstsci            

In [11]:
import numpy as np, random
dfp['lawobey'] = np.nan

In [12]:
import pandas as pd
import numpy as np

datahun = {
    'lawobey': {
        1: 0.60,
        2: 0.326,
        3: 0.062,
        4: 0.011,
        5: 0.002,
    },
    'trstsci': {
        0: 0.011,
        1: 0.01,
        2: 0.029,
        3: 0.03,
        4: 0.053,
        5: 0.176,
        6: 0.121,
        7: 0.177,
        8: 0.197,
        9: 0.11,
        10: 0.086,
    }
}

def rngpond(df, dataspa, cntry_value, missing_threshold):

    df_cntry = df[df['cntry'] == cntry_value].copy()  

    for columna, pesos in dataspa.items():
        if columna in df_cntry.columns and df_cntry[columna].isnull().any():
            valores = np.array(list(pesos.keys()))
            pesos_ponderados = np.array(list(pesos.values()))

            
            suma_pesos = np.sum(pesos_ponderados)
            pesos_normalizados = pesos_ponderados / suma_pesos

            for index, row in df_cntry[df_cntry[columna].isnull()].iterrows():
                
                valor_aleatorio = np.random.choice(valores, p=pesos_normalizados)

                
                df_cntry.loc[index, columna] = valor_aleatorio

    
    for columna in df_cntry.columns:
        if df_cntry[columna].isnull().any():
            missing_percentage = df_cntry[columna].isnull().sum() / len(df_cntry)
            if missing_percentage < missing_threshold:
                mode_value = df_cntry[columna].mode()[0]
                df_cntry[columna].fillna(mode_value, inplace=True)

    
    df.loc[df['cntry'] == cntry_value] = df_cntry 


rngpond(dfp, datahun, 'Hungría', missing_threshold=0.2)
print(dfp[dfp['cntry'] == 'Hungría'].isnull().sum())

cntry                   0
pplfair                 0
pplhlp                  0
ppltrst                 0
lrscale                 0
polintr                 0
stfdem                  0
stfeco                  0
stfgov                  0
trstep                  0
trstlgl                 0
trstplc                 0
trstplt                 0
trstprl                 0
trstprt                 0
trstsci                 0
imbgeco                 0
imwbcnt                 0
happy                   0
rlgdgr                  0
cntrycat_factorizado    0
lawobey                 0
dtype: int64


C:\Users\Josue\AppData\Local\Temp\ipykernel_9372\1506205070.py:53: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cntry[columna].fillna(mode_value, inplace=True)


In [13]:
datask = {
    'lawobey': {
        1: 0.50,
        2: 0.4,
        3: 0.080,
        4: 0.014,
        5: 0.006,
    },
    'trstsci': {
        0: 0.047,
        1: 0.022,
        2: 0.04,
        3: 0.062,
        4: 0.069,
        5: 0.136,
        6: 0.078,
        7: 0.132,
        8: 0.141,
        9: 0.102,
        10: 0.171,
    }
}
rngpond(dfp, datask, 'Eslovaquia', missing_threshold=0.2)
print(dfp[dfp['cntry'] == 'Eslovaquia'].isnull().sum())

cntry                   0
pplfair                 0
pplhlp                  0
ppltrst                 0
lrscale                 0
polintr                 0
stfdem                  0
stfeco                  0
stfgov                  0
trstep                  0
trstlgl                 0
trstplc                 0
trstplt                 0
trstprl                 0
trstprt                 0
trstsci                 0
imbgeco                 0
imwbcnt                 0
happy                   0
rlgdgr                  0
cntrycat_factorizado    0
lawobey                 0
dtype: int64


In [14]:
dfp.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31494 entries, 0 to 31493
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   cntry                 31494 non-null  object 
 1   pplfair               31494 non-null  int64  
 2   pplhlp                31494 non-null  int64  
 3   ppltrst               31494 non-null  int64  
 4   lrscale               31494 non-null  int64  
 5   polintr               31494 non-null  int64  
 6   stfdem                31494 non-null  int64  
 7   stfeco                31494 non-null  int64  
 8   stfgov                31494 non-null  int64  
 9   trstep                31494 non-null  int64  
 10  trstlgl               31494 non-null  int64  
 11  trstplc               31494 non-null  int64  
 12  trstplt               31494 non-null  int64  
 13  trstprl               31494 non-null  int64  
 14  trstprt               31494 non-null  float64
 15  trstsci            

In [15]:
#Eliminamos los valores nulos de las columnas, que hacen referencia a que el encuestado no ha querido responder. valores 77,88,99
import pandas as pd
import numpy as np

def delnulos(df):

    columnas_a_limpiar = ['pplfair','pplhlp', 'ppltrst', 'lrscale', 'stfdem','stfeco','stfgov', 'trstep', 'trstlgl', 'trstplc', 'trstplt', 'trstprl', 'trstprt','trstsci','imbgeco','imwbcnt','happy','rlgdgr']
    for columna in columnas_a_limpiar:
        if columna in df.columns:
            df[columna] = np.where(df[columna] > 70, np.nan, df[columna])
            df.dropna(subset=[columna], inplace=True)
    col_v2 = ['polintr']
    for col in col_v2:
        if col in df.columns:
            df[col] = np.where(df[col] > 4, np.nan, df[col])
            df.dropna(subset=[col], inplace=True)
    return df

delnulos(dfp)
print(dfp.isnull().sum())

cntry                   0
pplfair                 0
pplhlp                  0
ppltrst                 0
lrscale                 0
polintr                 0
stfdem                  0
stfeco                  0
stfgov                  0
trstep                  0
trstlgl                 0
trstplc                 0
trstplt                 0
trstprl                 0
trstprt                 0
trstsci                 0
imbgeco                 0
imwbcnt                 0
happy                   0
rlgdgr                  0
cntrycat_factorizado    0
lawobey                 0
dtype: int64


In [16]:
dfp.rename(columns={'lawobey': 'lw_pnd', 'trstsci': 'trtsci_pnd','cntrycat_factorizado': 'cntgrp_fc',}, inplace=True)

In [18]:
dfp.to_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/raw/UEporG2.csv',index=False)

In [19]:
%reset -f

AHORA VAMOS A UNIFICAR TODOS LOS DATAFRAMES EN UNO PARA HACER UN ANALISIS DESCRIPTIVO DE LAS VARIABLES, ADEMAS DE FACTORIZAR EL PAIS.

In [21]:
import numpy as np, random
import pandas as pd

In [24]:
dfr = pd.read_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/raw/UEricG0.csv')
dfm = pd.read_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/raw/UEmedG1.csv')
dfp = pd.read_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/raw/UEporG2.csv')

In [27]:
dfue= pd.concat([dfr, dfm, dfp], ignore_index=True)

In [28]:
#Factorizamos la variable country por orden de aparicion.
valores_unicos = dfue['cntry'].unique()
mapeo_cntry = {valor: indice for indice, valor in enumerate(valores_unicos)}
dfue['cnt_fc'] = dfue['cntry'].map(mapeo_cntry)
print(dfue[['cntry', 'cnt_fc']])

             cntry  cnt_fc
0          Bélgica       0
1          Bélgica       0
2          Bélgica       0
3          Bélgica       0
4          Bélgica       0
...            ...     ...
128558  Eslovaquia       8
128559  Eslovaquia       8
128560  Eslovaquia       8
128561  Eslovaquia       8
128562  Eslovaquia       8

[128563 rows x 2 columns]


In [29]:
dfue.to_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/UE128k.csv', index=False)

Empezamos el paso 3 en el siguiente documento. Almacenaremos Todos los datos interesantes de nuestras bases de datos locales, dandoles el tipo de formato si fuera necesario al valor de las columnas, para tener una copia de seguridad en un servidor mediante POSTGRESSQL. Intentaremos alojar las tablas de EcoUE.db fuera del proyecto.